# **Training**
Complete this notebook to train a Pitch Detection Model and Player Detection Model using Yolo v8 and deploy your final model to Roboflow. 

---

## Preparation

**Imports**

In [ ]:
from roboflow import Roboflow

**Roboflow API Key**

Before running this block, make sure your Roboflow API keys are stored in `./env/keys.env` as specifed in the [Getting Started](https://github.com/JohnComonitski/FootballTrackingDataGeneration?tab=readme-ov-file#getting-started) section of this repo.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("./../env/keys.env")

# Access the variables
ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API")
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

In [ ]:
%mkdir datasets
%cd ./datasets

---

## Training and Deploying a Pitch Detection Model

A pitch detection model will be trained using a dataset provided by Roboflow. The train dataset can be found [here](https://universe.roboflow.com/roboflow-jvuqo/football-field-detection-f07vi) and we will begin by forking the dataset. Open the data set and click the ***Fork*** button. 

![fork.png](./../examples/fork.png)

Once forked, you will be prompted with a download button, click the download button.

![download.png](./../examples/download.png)

You will now be prompted to select your model type, for this proejct I used YOLOv8. Additionally, check the ***Show download code*** button and uncheck the ***Also train a model for Lable Assist with Roboflow Train*** option, since we will be training the model ourselves.

![ascode.png](./../examples/ascode.png)

Finally, the dataset will be forked into your own project within your Roboflow workspace. This will be the project we train and deploy our model to.

![code.png](./../examples/code.png)

You will be given code to download your dataset. Replace the code below, by copying and pasting the last 3 lines of Python code into the cell below.

In [ ]:
project = rf.workspace("johncomonitski").project("football-field-detection-f07vi-yukgc")
version = project.version(1)
dataset = version.download("yolov8")

Before running the next cell, update the numbers to match the version number indicated by this line of code `project.version(1)` given to you.

In [ ]:
!sed -i '' 's|\(train: \).*|\1../train/images|' "./football-field-detection-1/data.yaml"
!sed -i '' 's|\(val: \).*|\1../valid/images|' "./football-field-detection-1/data.yaml"

**Pitch Detection Training**

In [ ]:
%cd ..

!yolo task=pose mode=train model=yolov8x-pose.pt data={dataset.location}/data.yaml batch=6 epochs=100 imgsz=1280 mosaic=0.0 plots=True project=./../models name=pitch_detection_model

**Pitch Detection Deployment**

In [ ]:
project.version(dataset.version).deploy(model_type="yolov8-pose", model_path=f"./../../models/pitch_detection_model")

---

## Training and Deploying a Player Detection Model

Player detection training will follow the same exact process as training our pitch detection model. We will start by using [this](https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc) dataset provided by Roboflow. This dataset is used to train a model to detect the ball, players and referees. Open that link and follow the exact same forking process as the last model. This means forking the dataset, downloading the dataset as code and copying the code snippet into the cell below.

In [ ]:
project = rf.workspace("johncomonitski").project("football-players-detection-3zvbc-btky1")
version = project.version(1)
dataset = version.download("yolov8")

Before running the next cell, update the numbers to match the version number indicated by this line of code `project.version(1)` given to you.

In [ ]:
!sed -i '' 's|\(train: \).*|\1../train/images|' "./football-players-detection-1/data.yaml"
!sed -i '' 's|\(val: \).*|\1../valid/images|' "./football-players-detection-1/data.yaml"

**Player Detection Training**

In [ ]:
!yolo task=detect mode=train model=yolov8x.pt data={dataset.location}/data.yaml batch=6 epochs=50 imgsz=1280 plots=True project=./../models name=player_detection_model

**Player Detection Deployment**

In [ ]:
project.version(dataset.version).deploy(model_type="yolov8", model_path=f"./../../models/player_detection_model/")